In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import random

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


In [3]:
train_data = pd.read_csv('archive4/mnist_train.csv')
test_data = pd.read_csv('archive4/mnist_test.csv')

X_train = np.asarray(train_data.iloc[:, 1:]).astype('float32')
y_train = np.asarray(train_data['label']).astype('int64')
X_test = np.asarray(test_data.iloc[:, 1:]).astype('float32')
y_test = np.asarray(test_data['label']).astype('int64')

X_train_t = torch.tensor(X_train, device=device)
y_train_t = torch.tensor(y_train, device=device)
X_test_t = torch.tensor(X_test, device=device)
y_test_t = torch.tensor(y_test, device=device)

batch_size = 256
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [4]:
class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(512, 512)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop2(x)
        x = self.fc3(x)
        return x

In [5]:
model_mnist = MNISTModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mnist.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    model_mnist.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model_mnist(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}')

model_mnist.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model_mnist(batch_X)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
accuracy_mnist = correct / total
print(f'MNIST test accuracy: {accuracy_mnist:.4f}')

Epoch 1/50, Loss: 0.6445
Epoch 2/50, Loss: 0.1677
Epoch 3/50, Loss: 0.1253
Epoch 4/50, Loss: 0.1062
Epoch 5/50, Loss: 0.0952
Epoch 6/50, Loss: 0.0877
Epoch 7/50, Loss: 0.0814
Epoch 8/50, Loss: 0.0796
Epoch 9/50, Loss: 0.0777
Epoch 10/50, Loss: 0.0774
Epoch 11/50, Loss: 0.0847
Epoch 12/50, Loss: 0.0773
Epoch 13/50, Loss: 0.0684
Epoch 14/50, Loss: 0.0782
Epoch 15/50, Loss: 0.0778
Epoch 16/50, Loss: 0.0820
Epoch 17/50, Loss: 0.0886
Epoch 18/50, Loss: 0.0762
Epoch 19/50, Loss: 0.0873
Epoch 20/50, Loss: 0.0741
Epoch 21/50, Loss: 0.0789
Epoch 22/50, Loss: 0.0692
Epoch 23/50, Loss: 0.0876
Epoch 24/50, Loss: 0.0762
Epoch 25/50, Loss: 0.0658
Epoch 26/50, Loss: 0.0725
Epoch 27/50, Loss: 0.0753
Epoch 28/50, Loss: 0.0731
Epoch 29/50, Loss: 0.0777
Epoch 30/50, Loss: 0.0733
Epoch 31/50, Loss: 0.0790
Epoch 32/50, Loss: 0.0738
Epoch 33/50, Loss: 0.0700
Epoch 34/50, Loss: 0.0761
Epoch 35/50, Loss: 0.0660
Epoch 36/50, Loss: 0.0654
Epoch 37/50, Loss: 0.0652
Epoch 38/50, Loss: 0.0672
Epoch 39/50, Loss: 0.

In [6]:
torch.save(model_mnist.state_dict(), 'mnist_weights.pth')
print("Веса сохранены в mnist_weights.pth")

Веса сохранены в mnist_weights.pth


In [7]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

train_set_fashion = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)
test_set_fashion = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader_fashion = DataLoader(train_set_fashion, batch_size=batch_size, shuffle=True)
test_loader_fashion = DataLoader(test_set_fashion, batch_size=batch_size, shuffle=False)

model_fashion = MNISTModel().to(device)

model_fashion.load_state_dict(torch.load('mnist_weights.pth', map_location=device))
print("Веса из MNIST загружены.")


Веса из MNIST загружены.


In [10]:
epochs_frozen = 15
epochs_unfrozen = 10
total_epochs = epochs_frozen + epochs_unfrozen

for name, param in model_fashion.named_parameters():
    if name.startswith('fc3'):
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model_fashion.parameters()), lr=0.001)

criterion = nn.CrossEntropyLoss()

for epoch in range(total_epochs):
    if epoch == epochs_frozen:
        print("Размораживаем все слои для дальнейшего обучения...")
        for param in model_fashion.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model_fashion.parameters(), lr=0.0005)

    model_fashion.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader_fashion:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_fashion(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_loader_fashion.dataset)
    print(f'Epoch {epoch+1}/{total_epochs}, Loss: {epoch_loss:.4f}')

model_fashion.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_X, batch_y in test_loader_fashion:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = model_fashion(batch_X)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
accuracy_fashion = correct / total
print(f'Fashion-MNIST test accuracy after transfer: {accuracy_fashion:.4f}')

Epoch 1/25, Loss: 0.3696
Epoch 2/25, Loss: 0.3637
Epoch 3/25, Loss: 0.3627
Epoch 4/25, Loss: 0.3625
Epoch 5/25, Loss: 0.3609
Epoch 6/25, Loss: 0.3615
Epoch 7/25, Loss: 0.3620
Epoch 8/25, Loss: 0.3604
Epoch 9/25, Loss: 0.3594
Epoch 10/25, Loss: 0.3583
Epoch 11/25, Loss: 0.3583
Epoch 12/25, Loss: 0.3565
Epoch 13/25, Loss: 0.3578
Epoch 14/25, Loss: 0.3580
Epoch 15/25, Loss: 0.3574
Размораживаем все слои для дальнейшего обучения...
Epoch 16/25, Loss: 0.3642
Epoch 17/25, Loss: 0.3508
Epoch 18/25, Loss: 0.3355
Epoch 19/25, Loss: 0.3242
Epoch 20/25, Loss: 0.3117
Epoch 21/25, Loss: 0.3059
Epoch 22/25, Loss: 0.2962
Epoch 23/25, Loss: 0.2899
Epoch 24/25, Loss: 0.2860
Epoch 25/25, Loss: 0.2773
Fashion-MNIST test accuracy after transfer: 0.8834


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class FlexibleModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(512, 512)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop2(x)
        x = self.fc3(x)
        return x

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

fashion_full_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_full_test  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

selected_classes = [0, 1, 2, 3, 4]
fashion5_train_indices = [i for i, (_, label) in enumerate(fashion_full_train) if label in selected_classes]
fashion5_test_indices  = [i for i, (_, label) in enumerate(fashion_full_test) if label in selected_classes]

fashion5_train = Subset(fashion_full_train, fashion5_train_indices)
fashion5_test  = Subset(fashion_full_test, fashion5_test_indices)

In [4]:
batch_size = 256
train_loader_mnist = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader_mnist  = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)
train_loader_fashion5 = DataLoader(fashion5_train, batch_size=batch_size, shuffle=True)
test_loader_fashion5  = DataLoader(fashion5_test, batch_size=batch_size, shuffle=False)

def train_model(model, train_loader, epochs, lr=0.001, freeze_layers=None, unfreeze_epoch=None):
    model.to(device)
    criterion = nn.CrossEntropyLoss()

    if freeze_layers:
        for name, param in model.named_parameters():
            if any(name.startswith(layer) for layer in freeze_layers):
                param.requires_grad = False
            else:
                param.requires_grad = True
    else:
        for param in model.parameters():
            param.requires_grad = True

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    for epoch in range(epochs):
        if unfreeze_epoch is not None and epoch == unfreeze_epoch:
            print(f"Unfreezing all layers at epoch {epoch}")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.Adam(model.parameters(), lr=lr * 0.5)  # уменьшаем LR

        model.train()
        running_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_X.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}')
    return model

In [5]:
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    return correct / total

In [6]:
def copy_body_weights(source_model, target_model):
    with torch.no_grad():
        target_model.fc1.weight.copy_(source_model.fc1.weight)
        target_model.fc1.bias.copy_(source_model.fc1.bias)
        target_model.fc2.weight.copy_(source_model.fc2.weight)
        target_model.fc2.bias.copy_(source_model.fc2.bias)

In [17]:
model_mnist = FlexibleModel(num_classes=10)
model_mnist = train_model(model_mnist, train_loader_mnist, epochs=40, lr=0.001)
acc_mnist = evaluate(model_mnist, test_loader_mnist)
print(f"MNIST test accuracy: {acc_mnist:.4f}")
torch.save(model_mnist.state_dict(), 'mnist_10_weights.pth')

Epoch 1/40, Loss: 0.3660
Epoch 2/40, Loss: 0.1330
Epoch 3/40, Loss: 0.0924
Epoch 4/40, Loss: 0.0687
Epoch 5/40, Loss: 0.0555
Epoch 6/40, Loss: 0.0451
Epoch 7/40, Loss: 0.0367
Epoch 8/40, Loss: 0.0316
Epoch 9/40, Loss: 0.0271
Epoch 10/40, Loss: 0.0254
Epoch 11/40, Loss: 0.0221
Epoch 12/40, Loss: 0.0181
Epoch 13/40, Loss: 0.0193
Epoch 14/40, Loss: 0.0170
Epoch 15/40, Loss: 0.0165
Epoch 16/40, Loss: 0.0172
Epoch 17/40, Loss: 0.0142
Epoch 18/40, Loss: 0.0136
Epoch 19/40, Loss: 0.0116
Epoch 20/40, Loss: 0.0134
Epoch 21/40, Loss: 0.0137
Epoch 22/40, Loss: 0.0105
Epoch 23/40, Loss: 0.0106
Epoch 24/40, Loss: 0.0140
Epoch 25/40, Loss: 0.0125
Epoch 26/40, Loss: 0.0105
Epoch 27/40, Loss: 0.0103
Epoch 28/40, Loss: 0.0091
Epoch 29/40, Loss: 0.0094
Epoch 30/40, Loss: 0.0095
Epoch 31/40, Loss: 0.0104
Epoch 32/40, Loss: 0.0080
Epoch 33/40, Loss: 0.0097
Epoch 34/40, Loss: 0.0115
Epoch 35/40, Loss: 0.0108
Epoch 36/40, Loss: 0.0079
Epoch 37/40, Loss: 0.0072
Epoch 38/40, Loss: 0.0095
Epoch 39/40, Loss: 0.

In [18]:
model_fashion5 = FlexibleModel(num_classes=5)
model_fashion5 = train_model(model_fashion5, train_loader_fashion5, epochs=40, lr=0.001)
acc_fashion5 = evaluate(model_fashion5, test_loader_fashion5)
print(f"Fashion5 test accuracy: {acc_fashion5:.4f}")
torch.save(model_fashion5.state_dict(), 'fashion5_weights.pth')

Epoch 1/40, Loss: 0.4711
Epoch 2/40, Loss: 0.3265
Epoch 3/40, Loss: 0.2946
Epoch 4/40, Loss: 0.2761
Epoch 5/40, Loss: 0.2597
Epoch 6/40, Loss: 0.2560
Epoch 7/40, Loss: 0.2424
Epoch 8/40, Loss: 0.2358
Epoch 9/40, Loss: 0.2289
Epoch 10/40, Loss: 0.2195
Epoch 11/40, Loss: 0.2114
Epoch 12/40, Loss: 0.2069
Epoch 13/40, Loss: 0.2064
Epoch 14/40, Loss: 0.1971
Epoch 15/40, Loss: 0.1900
Epoch 16/40, Loss: 0.1898
Epoch 17/40, Loss: 0.1816
Epoch 18/40, Loss: 0.1849
Epoch 19/40, Loss: 0.1787
Epoch 20/40, Loss: 0.1694
Epoch 21/40, Loss: 0.1659
Epoch 22/40, Loss: 0.1664
Epoch 23/40, Loss: 0.1640
Epoch 24/40, Loss: 0.1643
Epoch 25/40, Loss: 0.1564
Epoch 26/40, Loss: 0.1505
Epoch 27/40, Loss: 0.1476
Epoch 28/40, Loss: 0.1467
Epoch 29/40, Loss: 0.1426
Epoch 30/40, Loss: 0.1411
Epoch 31/40, Loss: 0.1423
Epoch 32/40, Loss: 0.1404
Epoch 33/40, Loss: 0.1379
Epoch 34/40, Loss: 0.1344
Epoch 35/40, Loss: 0.1327
Epoch 36/40, Loss: 0.1339
Epoch 37/40, Loss: 0.1275
Epoch 38/40, Loss: 0.1325
Epoch 39/40, Loss: 0.

In [19]:
print("\n--- Strategy A: freeze body, train only head ---")
model_transfer_A = FlexibleModel(num_classes=5)
copy_body_weights(model_mnist, model_transfer_A)
model_transfer_A = train_model(model_transfer_A, train_loader_fashion5, epochs=20, lr=0.001,
                               freeze_layers=['fc1','fc2'], unfreeze_epoch=None)
acc_A = evaluate(model_transfer_A, test_loader_fashion5)
print(f"Fashion5 test accuracy (only head): {acc_A:.4f}")


--- Strategy A: freeze body, train only head ---
Epoch 1/20, Loss: 0.9203
Epoch 2/20, Loss: 0.7082
Epoch 3/20, Loss: 0.6579
Epoch 4/20, Loss: 0.6238
Epoch 5/20, Loss: 0.6088
Epoch 6/20, Loss: 0.5994
Epoch 7/20, Loss: 0.5900
Epoch 8/20, Loss: 0.5827
Epoch 9/20, Loss: 0.5749
Epoch 10/20, Loss: 0.5739
Epoch 11/20, Loss: 0.5706
Epoch 12/20, Loss: 0.5630
Epoch 13/20, Loss: 0.5631
Epoch 14/20, Loss: 0.5603
Epoch 15/20, Loss: 0.5593
Epoch 16/20, Loss: 0.5526
Epoch 17/20, Loss: 0.5550
Epoch 18/20, Loss: 0.5555
Epoch 19/20, Loss: 0.5524
Epoch 20/20, Loss: 0.5527
Fashion5 test accuracy (only head): 0.8058


In [21]:
print("\n--- Strategy B: head first, then unfreeze body after 10 epochs ---")
model_transfer_B = FlexibleModel(num_classes=5)
copy_body_weights(model_mnist, model_transfer_B)
model_transfer_B = train_model(model_transfer_B, train_loader_fashion5, epochs=20, lr=0.001,
                               freeze_layers=['fc1','fc2'], unfreeze_epoch=10)
acc_B = evaluate(model_transfer_B, test_loader_fashion5)
print(f"Fashion5 test accuracy (head + unfreeze body): {acc_B:.4f}")


--- Strategy B: head first, then unfreeze body after 10 epochs ---
Epoch 1/20, Loss: 0.9031
Epoch 2/20, Loss: 0.7002
Epoch 3/20, Loss: 0.6524
Epoch 4/20, Loss: 0.6242
Epoch 5/20, Loss: 0.6064
Epoch 6/20, Loss: 0.5975
Epoch 7/20, Loss: 0.5900
Epoch 8/20, Loss: 0.5815
Epoch 9/20, Loss: 0.5754
Epoch 10/20, Loss: 0.5720
Unfreezing all layers at epoch 10
Epoch 11/20, Loss: 0.4404
Epoch 12/20, Loss: 0.3501
Epoch 13/20, Loss: 0.3092
Epoch 14/20, Loss: 0.2869
Epoch 15/20, Loss: 0.2691
Epoch 16/20, Loss: 0.2521
Epoch 17/20, Loss: 0.2420
Epoch 18/20, Loss: 0.2319
Epoch 19/20, Loss: 0.2217
Epoch 20/20, Loss: 0.2133
Fashion5 test accuracy (head + unfreeze body): 0.8998


In [22]:
print("\n--- Strategy A: freeze body, train only head ---")
model_transfer_C = FlexibleModel(num_classes=10)
copy_body_weights(model_fashion5, model_transfer_C)
model_transfer_C = train_model(model_transfer_C, train_loader_mnist, epochs=20, lr=0.001,
                               freeze_layers=['fc1','fc2'], unfreeze_epoch=None)
acc_C = evaluate(model_transfer_C, test_loader_mnist)
print(f"MNIST test accuracy (only head): {acc_C:.4f}")


--- Strategy A: freeze body, train only head ---
Epoch 1/20, Loss: 1.3958
Epoch 2/20, Loss: 1.0373
Epoch 3/20, Loss: 0.9374
Epoch 4/20, Loss: 0.8869
Epoch 5/20, Loss: 0.8549
Epoch 6/20, Loss: 0.8359
Epoch 7/20, Loss: 0.8217
Epoch 8/20, Loss: 0.8122
Epoch 9/20, Loss: 0.7979
Epoch 10/20, Loss: 0.7915
Epoch 11/20, Loss: 0.7872
Epoch 12/20, Loss: 0.7812
Epoch 13/20, Loss: 0.7774
Epoch 14/20, Loss: 0.7730
Epoch 15/20, Loss: 0.7683
Epoch 16/20, Loss: 0.7609
Epoch 17/20, Loss: 0.7639
Epoch 18/20, Loss: 0.7634
Epoch 19/20, Loss: 0.7592
Epoch 20/20, Loss: 0.7598
MNIST test accuracy (only head): 0.8380


In [23]:
print("\n--- Strategy B: head first, then unfreeze body after 10 epochs ---")
model_transfer_D = FlexibleModel(num_classes=10)
copy_body_weights(model_fashion5, model_transfer_D)
model_transfer_D = train_model(model_transfer_D, train_loader_mnist, epochs=20, lr=0.001,
                               freeze_layers=['fc1','fc2'], unfreeze_epoch=10)
acc_D = evaluate(model_transfer_D, test_loader_mnist)
print(f"MNIST test accuracy (head + unfreeze body): {acc_D:.4f}")


--- Strategy B: head first, then unfreeze body after 10 epochs ---
Epoch 1/20, Loss: 1.3841
Epoch 2/20, Loss: 1.0354
Epoch 3/20, Loss: 0.9376
Epoch 4/20, Loss: 0.8905
Epoch 5/20, Loss: 0.8571
Epoch 6/20, Loss: 0.8351
Epoch 7/20, Loss: 0.8221
Epoch 8/20, Loss: 0.8105
Epoch 9/20, Loss: 0.8000
Epoch 10/20, Loss: 0.7946
Unfreezing all layers at epoch 10
Epoch 11/20, Loss: 0.3223
Epoch 12/20, Loss: 0.1504
Epoch 13/20, Loss: 0.1028
Epoch 14/20, Loss: 0.0775
Epoch 15/20, Loss: 0.0648
Epoch 16/20, Loss: 0.0514
Epoch 17/20, Loss: 0.0426
Epoch 18/20, Loss: 0.0374
Epoch 19/20, Loss: 0.0318
Epoch 20/20, Loss: 0.0283
MNIST test accuracy (head + unfreeze body): 0.9813


In [24]:
print("\n" + "="*50)
print("SUMMARY OF TRANSFER LEARNING RESULTS")
print("="*50)
print(f"MNIST baseline (10 classes): {acc_mnist:.4f}")
print(f"Fashion5 baseline (5 classes): {acc_fashion5:.4f}")
print("\nMNIST -> Fashion5:")
print(f"  Only head: {acc_A:.4f}")
print(f"  Head + unfreeze body: {acc_B:.4f}")
print("\nFashion5 -> MNIST:")
print(f"  Only head: {acc_C:.4f}")
print(f"  Head + unfreeze body: {acc_D:.4f}")


SUMMARY OF TRANSFER LEARNING RESULTS
MNIST baseline (10 classes): 0.9839
Fashion5 baseline (5 classes): 0.9122

MNIST -> Fashion5:
  Only head: 0.8058
  Head + unfreeze body: 0.8998

Fashion5 -> MNIST:
  Only head: 0.8380
  Head + unfreeze body: 0.9813


TensorFlow

In [25]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist, fashion_mnist
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [26]:
def load_fashion5(classes=[0,1,2,3,4]):
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    train_mask = np.isin(y_train, classes)
    test_mask = np.isin(y_test, classes)
    x_train, y_train = x_train[train_mask], y_train[train_mask]
    x_test, y_test = x_test[test_mask], y_test[test_mask]
    return (x_train, y_train), (x_test, y_test)

(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = mnist.load_data()
x_train_mnist, x_val_mnist, y_train_mnist, y_val_mnist = train_test_split(
    x_train_mnist, y_train_mnist, test_size=0.2, random_state=42
)

(x_train_f5, y_train_f5), (x_test_f5, y_test_f5) = load_fashion5(classes=[0,1,2,3,4])
x_train_f5, x_val_f5, y_train_f5, y_val_f5 = train_test_split(
    x_train_f5, y_train_f5, test_size=0.2, random_state=42
)

def preprocess(x):
    x = x.astype('float32') / 255.0
    if len(x.shape) == 3:
        x = x[..., None]
    return x

x_train_mnist = preprocess(x_train_mnist)
x_val_mnist = preprocess(x_val_mnist)
x_test_mnist = preprocess(x_test_mnist)
x_train_f5 = preprocess(x_train_f5)
x_val_f5 = preprocess(x_val_f5)
x_test_f5 = preprocess(x_test_f5)

print(f"MNIST train: {x_train_mnist.shape}, val: {x_val_mnist.shape}")
print(f"Fashion5 train: {x_train_f5.shape}, val: {x_val_f5.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
MNIST train: (48000, 28, 28, 1), val: (12000, 28, 28, 1)
Fashion5 train: (24000, 28, 28, 1), val: (6000, 28, 28, 1)


In [27]:
def create_cnn(input_shape=(28,28,1), num_classes=10, filters1=32, filters2=64, kernel_size=3, dropout_rate=0.5):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(filters1, kernel_size, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(dropout_rate),
        
        layers.Conv2D(filters2, kernel_size, padding='same'),
        layers.LayerNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),
        
        layers.Flatten(),
        layers.Dense(128),
        layers.UnitNormalization(axis=-1),
        layers.Activation('relu'),
        layers.Dropout(dropout_rate),
        
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

In [28]:
def train_model(model, x_train, y_train, x_val, y_val, epochs=20, lr=0.001, patience=5, verbose=1):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=epochs,
                        batch_size=128,
                        callbacks=[early_stop],
                        verbose=verbose)
    return history

def evaluate(model, x_test, y_test):
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    return acc

def copy_body_weights(source_model, target_model, num_layers_to_copy=None):
    if num_layers_to_copy is None:
        num_layers_to_copy = len(source_model.layers) - 1
    for i in range(min(num_layers_to_copy, len(source_model.layers), len(target_model.layers))):
        if i < num_layers_to_copy:  # копируем только тело
            target_model.layers[i].set_weights(source_model.layers[i].get_weights())
    return target_model

In [37]:
print("\n=== Training base model on MNIST (10 classes) ===")
model_mnist = create_cnn(num_classes=10)
history_mnist = train_model(model_mnist, x_train_mnist, y_train_mnist, x_val_mnist, y_val_mnist, epochs=40)
acc_mnist = evaluate(model_mnist, x_test_mnist, y_test_mnist)
print(f"MNIST test accuracy: {acc_mnist:.4f}")
model_mnist.save('mnist_10_model.h5')


=== Training base model on MNIST (10 classes) ===
Epoch 1/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 39s 99ms/step - accuracy: 0.8651 - loss: 1.1390 - val_accuracy: 0.9691 - val_loss: 0.5003
Epoch 2/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 45s 121ms/step - accuracy: 0.9667 - loss: 0.3780 - val_accuracy: 0.9797 - val_loss: 0.2024
Epoch 3/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 44s 118ms/step - accuracy: 0.9740 - loss: 0.2097 - val_accuracy: 0.9842 - val_loss: 0.1168
Epoch 4/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 44s 118ms/step - accuracy: 0.9796 - loss: 0.1412 - val_accuracy: 0.9868 - val_loss: 0.0814
Epoch 5/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 44s 118ms/step - accuracy: 0.9819 - loss: 0.1105 - val_accuracy: 0.9862 - val_loss: 0.0737
Epoch 6/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 45s 119ms/step - accuracy: 0.9840 - loss: 0.0914 - val_accuracy: 0.9880 - val_loss: 0.0562
Epoch 7/40
375/375 ━━━━━━━━━━━━━━━━━━━━ 45s 119ms/step - accuracy: 0.9854 - loss: 0.0790 - val_accuracy: 0.9887 - val_loss: 0.0476
Epoch 8/40
375/375 ━━━━━━━━━━━━━━

MNIST test accuracy: 0.9941


In [38]:
print("\n=== Training base model on Fashion5 (5 classes) ===")
model_f5 = create_cnn(num_classes=5)
history_f5 = train_model(model_f5, x_train_f5, y_train_f5, x_val_f5, y_val_f5, epochs=40)
acc_f5 = evaluate(model_f5, x_test_f5, y_test_f5)
print(f"Fashion5 test accuracy: {acc_f5:.4f}")
model_f5.save('fashion5_model.h5')


=== Training base model on Fashion5 (5 classes) ===
Epoch 1/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 26s 118ms/step - accuracy: 0.7906 - loss: 0.8812 - val_accuracy: 0.8743 - val_loss: 0.5822
Epoch 2/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 117ms/step - accuracy: 0.8906 - loss: 0.4795 - val_accuracy: 0.9102 - val_loss: 0.3727
Epoch 3/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 116ms/step - accuracy: 0.9055 - loss: 0.3606 - val_accuracy: 0.9182 - val_loss: 0.2940
Epoch 4/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 117ms/step - accuracy: 0.9163 - loss: 0.3005 - val_accuracy: 0.9248 - val_loss: 0.2601
Epoch 5/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 116ms/step - accuracy: 0.9201 - loss: 0.2696 - val_accuracy: 0.9343 - val_loss: 0.2152
Epoch 6/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 116ms/step - accuracy: 0.9225 - loss: 0.2478 - val_accuracy: 0.9375 - val_loss: 0.2065
Epoch 7/40
188/188 ━━━━━━━━━━━━━━━━━━━━ 22s 116ms/step - accuracy: 0.9283 - loss: 0.2312 - val_accuracy: 0.9193 - val_loss: 0.2234
Epoch 8/40
188/188 ━━━━━━━━━━━

Fashion5 test accuracy: 0.9434


In [39]:
print("\n--- Strategy A: train only new head (body frozen) ---")
model_transfer_A = create_cnn(num_classes=5)
copy_body_weights(model_mnist, model_transfer_A)
for layer in model_transfer_A.layers[:-1]:
    layer.trainable = False
history_A = train_model(model_transfer_A, x_train_f5, y_train_f5, x_val_f5, y_val_f5, epochs=20, lr=0.001, verbose=1)
acc_A = evaluate(model_transfer_A, x_test_f5, y_test_f5)
print(f"Fashion5 test accuracy (only head): {acc_A:.4f}")


--- Strategy A: train only new head (body frozen) ---
Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.4733 - loss: 1.4674 - val_accuracy: 0.6707 - val_loss: 1.3230
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.6150 - loss: 1.2548 - val_accuracy: 0.6935 - val_loss: 1.1437
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.6437 - loss: 1.1225 - val_accuracy: 0.6975 - val_loss: 1.0266
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.6577 - loss: 1.0335 - val_accuracy: 0.7127 - val_loss: 0.9453
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.6687 - loss: 0.9773 - val_accuracy: 0.7177 - val_loss: 0.8875
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.6702 - loss: 0.9352 - val_accuracy: 0.7203 - val_loss: 0.8444
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.6787 - loss: 0.9041 - val_accuracy: 0.7215 - val_loss: 0.8114
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s

In [40]:
print("\n--- Strategy B: head first, then unfreeze body after 10 epochs ---")
model_transfer_B = create_cnn(num_classes=5)
copy_body_weights(model_mnist, model_transfer_B)
for layer in model_transfer_B.layers[:-1]:
    layer.trainable = False
model_transfer_B.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                         loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_transfer_B.fit(x_train_f5, y_train_f5, validation_data=(x_val_f5, y_val_f5),
                     epochs=10, batch_size=128, verbose=1)
for layer in model_transfer_B.layers:
    layer.trainable = True
model_transfer_B.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                         loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_B = model_transfer_B.fit(x_train_f5, y_train_f5, validation_data=(x_val_f5, y_val_f5),
                                 epochs=10, batch_size=128, verbose=1)
acc_B = evaluate(model_transfer_B, x_test_f5, y_test_f5)
print(f"Fashion5 test accuracy (head + unfreeze): {acc_B:.4f}")


--- Strategy B: head first, then unfreeze body after 10 epochs ---
Epoch 1/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 34ms/step - accuracy: 0.4985 - loss: 1.4599 - val_accuracy: 0.6323 - val_loss: 1.3190
Epoch 2/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.6152 - loss: 1.2494 - val_accuracy: 0.6900 - val_loss: 1.1406
Epoch 3/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.6395 - loss: 1.1206 - val_accuracy: 0.6992 - val_loss: 1.0246
Epoch 4/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.6586 - loss: 1.0309 - val_accuracy: 0.7098 - val_loss: 0.9442
Epoch 5/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.6723 - loss: 0.9720 - val_accuracy: 0.7167 - val_loss: 0.8862
Epoch 6/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.6713 - loss: 0.9345 - val_accuracy: 0.7183 - val_loss: 0.8433
Epoch 7/10
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.6791 - loss: 0.9013 - val_accuracy: 0.7247 - val_loss: 0.8101
Epoch 8/10
188/188 ━━━━━━━━━━

In [41]:
print("\n--- Strategy A: train only new head (body frozen) ---")
model_transfer_C = create_cnn(num_classes=10)
copy_body_weights(model_f5, model_transfer_C)
for layer in model_transfer_C.layers[:-1]:
    layer.trainable = False
history_C = train_model(model_transfer_C, x_train_mnist, y_train_mnist, x_val_mnist, y_val_mnist, epochs=20, lr=0.001)
acc_C = evaluate(model_transfer_C, x_test_mnist, y_test_mnist)
print(f"MNIST test accuracy (only head): {acc_C:.4f}")


--- Strategy A: train only new head (body frozen) ---
Epoch 1/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.3968 - loss: 2.0850 - val_accuracy: 0.5959 - val_loss: 1.8735
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.5344 - loss: 1.7794 - val_accuracy: 0.6443 - val_loss: 1.6163
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.5710 - loss: 1.5957 - val_accuracy: 0.6735 - val_loss: 1.4452
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.5883 - loss: 1.4726 - val_accuracy: 0.6866 - val_loss: 1.3223
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.5956 - loss: 1.3899 - val_accuracy: 0.7053 - val_loss: 1.2296
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.6069 - loss: 1.3267 - val_accuracy: 0.7149 - val_loss: 1.1573
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.6166 - loss: 1.2750 - val_accuracy: 0.7262 - val_loss: 1.0984
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━

In [42]:
print("\n--- Strategy B: head first, then unfreeze body after 10 epochs ---")
model_transfer_D = create_cnn(num_classes=10)
copy_body_weights(model_f5, model_transfer_D)
for layer in model_transfer_D.layers[:-1]:
    layer.trainable = False
model_transfer_D.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                         loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_transfer_D.fit(x_train_mnist, y_train_mnist, validation_data=(x_val_mnist, y_val_mnist),
                     epochs=10, batch_size=128, verbose=1)
for layer in model_transfer_D.layers:
    layer.trainable = True
model_transfer_D.compile(optimizer=tf.keras.optimizers.Adam(0.0001),
                         loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_D = model_transfer_D.fit(x_train_mnist, y_train_mnist, validation_data=(x_val_mnist, y_val_mnist),
                                 epochs=10, batch_size=128, verbose=1)
acc_D = evaluate(model_transfer_D, x_test_mnist, y_test_mnist)
print(f"MNIST test accuracy (head + unfreeze): {acc_D:.4f}")


--- Strategy B: head first, then unfreeze body after 10 epochs ---
Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.3862 - loss: 2.0834 - val_accuracy: 0.5958 - val_loss: 1.8726
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 9s 25ms/step - accuracy: 0.5313 - loss: 1.7804 - val_accuracy: 0.6373 - val_loss: 1.6177
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.5656 - loss: 1.5963 - val_accuracy: 0.6696 - val_loss: 1.4460
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.5863 - loss: 1.4748 - val_accuracy: 0.6891 - val_loss: 1.3233
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.6000 - loss: 1.3877 - val_accuracy: 0.7028 - val_loss: 1.2305
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.6110 - loss: 1.3261 - val_accuracy: 0.7185 - val_loss: 1.1576
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.6157 - loss: 1.2781 - val_accuracy: 0.7252 - val_loss: 1.1003
Epoch 8/10
375/375 ━━━━

In [43]:
print("\n" + "="*50)
print("SUMMARY OF TRANSFER LEARNING RESULTS (TensorFlow)")
print("="*50)
print(f"MNIST baseline (10 classes): {acc_mnist:.4f}")
print(f"Fashion5 baseline (5 classes): {acc_f5:.4f}")
print("\nMNIST -> Fashion5:")
print(f"  Only head: {acc_A:.4f}")
print(f"  Head + unfreeze body: {acc_B:.4f}")
print("\nFashion5 -> MNIST:")
print(f"  Only head: {acc_C:.4f}")
print(f"  Head + unfreeze body: {acc_D:.4f}")


SUMMARY OF TRANSFER LEARNING RESULTS (TensorFlow)
MNIST baseline (10 classes): 0.9941
Fashion5 baseline (5 classes): 0.9434

MNIST -> Fashion5:
  Only head: 0.7454
  Head + unfreeze body: 0.8934

Fashion5 -> MNIST:
  Only head: 0.8039
  Head + unfreeze body: 0.9778
